In [ ]:
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
client = bigquery.Client(project="sous-nos-pieds")

In [ ]:

# 1 Authentif

from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
import pandas as pd

client = bigquery.Client(project="sous-nos-pieds")

# 2. Cible multi-label: commune x type_peril.....

query_cible = """
SELECT
  LPAD(CAST(code_commune AS STRING), 5, '0') AS code_commune,
  type_peril,
  SUM(nb_reconnaissances) AS nb_reconnaissances
FROM `sous-nos-pieds.staging.catnat_geo_final_staging`
GROUP BY code_commune, type_peril
"""
df_cible = client.query(query_cible).to_dataframe()

cible_pivot = (
    df_cible.pivot_table(index="code_commune", columns="type_peril",
                          values="nb_reconnaissances", fill_value=0)
    .gt(0).astype(int)
    .add_prefix("risque_")
    .reset_index()
)

# 3. Features climatiques : agrégats SWI par commune.....

query_swi = """
SELECT
  LPAD(CAST(code_commune AS STRING), 5, '0') AS code_commune,
  AVG(SWI_UNIF_MENS) AS swi_moyen,
  STDDEV(SWI_UNIF_MENS) AS swi_variabilite,
  MIN(SWI_UNIF_MENS) AS swi_min,
  MAX(SWI_UNIF_MENS) AS swi_max,
  COUNTIF(SWI_UNIF_MENS < 0.3) AS nb_mois_secheresse
FROM `sous-nos-pieds.staging.swi_par_commune`
GROUP BY code_commune
"""
df_swi = client.query(query_swi).to_dataframe()


# 4. Référentiel géographique (département/région)..............................

query_geo = """
SELECT
  LPAD(CAST(code_commune_INSEE AS STRING), 5, '0') AS code_commune,
  code_departement,
  nom_departement
FROM `sous-nos-pieds.staging.geoloc_staging`
"""
df_geo = client.query(query_geo).to_dataframe()


# 5. Jointure finale.....................................................................

df_final = (
    cible_pivot
    .merge(df_swi, on="code_commune", how="left")
    .merge(df_geo, on="code_commune", how="left")
)

print("Dimensions de la table finale :", df_final.shape)
df_final.head()

# 6. Diagnostic qualité — à toujours vérifier.......................................

print("\n--- Valeurs manquantes par colonne ---")
print(df_final.isna().sum())

print("\n--- Taux de communes cible sans correspondance SWI ---")
taux_sans_swi = df_final["swi_moyen"].isna().mean() * 100
print(f"{taux_sans_swi:.1f}% des communes cible n'ont pas de données SWI")

print("\n--- Taux de communes cible sans correspondance géo ---")
taux_sans_geo = df_final["code_departement"].isna().mean() * 100
print(f"{taux_sans_geo:.1f}% des communes cible n'ont pas de département")

# Repère les communes non matchées pour inspection (souvent Corse/DOM-TOM)............................
communes_non_matchees = df_final[df_final["swi_moyen"].isna()]["code_commune"].head(20)
print("\nExemples de codes commune non matchés :")
print(communes_non_matchees.tolist())

Dimensions de la table finale : (27304, 16)

--- Valeurs manquantes par colonne ---
code_commune                       0
risque_Autres périls               0
risque_Avalanche                   0
risque_Inondations                 0
risque_Mouvement de terrain        0
risque_Sécheresse                  0
risque_Séisme                      0
risque_Tous périls                 0
risque_Vent cyclonique             0
swi_moyen                      20025
swi_variabilite                20025
swi_min                        20025
swi_max                        20025
nb_mois_secheresse             20025
code_departement                   4
nom_departement                    4
dtype: int64

--- Taux de communes cible sans correspondance SWI ---
73.3% des communes cible n'ont pas de données SWI

--- Taux de communes cible sans correspondance géo ---
0.0% des communes cible n'ont pas de département

Exemples de codes commune non matchés :
['01001', '01002', '01004', '01005', '01007', '01008', '010

In [ ]:
# Combien de communes distinctes existent dans swi_par_commune au total ?.........
query_check1 = """
SELECT COUNT(DISTINCT code_commune) AS nb_communes_distinctes
FROM `sous-nos-pieds.staging.swi_par_commune`
"""
print(client.query(query_check1).to_dataframe())

# Est-ce qu'il y a des communes du département 01 (Ain) dans swi_par_commune ?..........
query_check2 = """
SELECT DISTINCT code_commune
FROM `sous-nos-pieds.staging.swi_par_commune`
WHERE CAST(code_commune AS STRING) LIKE '1%'
   OR CAST(code_commune AS STRING) LIKE '01%'
LIMIT 20
"""
print(client.query(query_check2).to_dataframe())

# Compare avec le nombre total de communes attendues en France (~35000)...........
query_check3 = """
SELECT MIN(code_commune) AS min_code, MAX(code_commune) AS max_code
FROM `sous-nos-pieds.staging.swi_par_commune`
"""
print(client.query(query_check3).to_dataframe())

   nb_communes_distinctes
0                    8078
   code_commune
0         12194
1         18193
2         14701
3         12127
4         01124
5         12044
6         12145
7         19036
8         15207
9         14039
10        17322
11        13105
12        16079
13        13092
14        16168
15        15114
16        17003
17        01199
18        18277
19        16246
  min_code max_code
0    01006    95652


In [ ]:

# 7. Rattachement spatial commune ↔ grille SWI...........

import numpy as np
from pyproj import Transformer
from scipy.spatial import cKDTree

# --- 7a. Récupérer les points de grille SWI (uniques) + stats agrégées........
query_grille_swi = """
SELECT
  NUMERO,
  ANY_VALUE(LAMBX) AS lambx,
  ANY_VALUE(LAMBY) AS lamby,
  AVG(SWI_UNIF_MENS) AS swi_moyen,
  STDDEV(SWI_UNIF_MENS) AS swi_variabilite,
  MIN(SWI_UNIF_MENS) AS swi_min,
  MAX(SWI_UNIF_MENS) AS swi_max,
  COUNTIF(SWI_UNIF_MENS < 0.3) AS nb_mois_secheresse
FROM `sous-nos-pieds.staging.donnees_50_ans_swi_staging`
GROUP BY NUMERO
"""
df_grille = client.query(query_grille_swi).to_dataframe()
print("Points de grille SWI :", df_grille.shape)

# --- 7b. Convertir Lambert-93 (EPSG:2154) -> lat/long (EPSG:4326)............
transformer = Transformer.from_crs("EPSG:2154", "EPSG:4326", always_xy=True)
lon_grille, lat_grille = transformer.transform(df_grille["lambx"].values, df_grille["lamby"].values)
df_grille["lat_grille"] = lat_grille
df_grille["lon_grille"] = lon_grille

# --- 7c. Récupérer TOUTES les communes avec lat/long (référentiel complet)...........
query_communes_latlon = """
SELECT
  LPAD(CAST(code_commune_INSEE AS STRING), 5, '0') AS code_commune,
  latitude,
  longitude,
  code_departement,
  nom_departement
FROM `sous-nos-pieds.staging.geoloc_staging`
WHERE latitude IS NOT NULL AND longitude IS NOT NULL
"""
df_communes = client.query(query_communes_latlon).to_dataframe()
print("Communes avec géoloc :", df_communes.shape)

# --- 7d. Nearest neighbor : chaque commune -> point de grille SWI le plus proche ............
# (approximation degrés->distance suffisante à l'échelle d'une commune/grille ~8km)................
arbre_grille = cKDTree(df_grille[["lat_grille", "lon_grille"]].values)
distances, indices = arbre_grille.query(df_communes[["latitude", "longitude"]].values)

df_communes["numero_grille_proche"] = df_grille.iloc[indices]["NUMERO"].values
df_communes["distance_deg"] = distances  # ~0.01° ≈ 1km, utile pour repérer les valeurs aberrantes...........

# --- 7e. Joindre les stats SWI de la grille rattachée à chaque commune...............
df_communes_swi = df_communes.merge(
    df_grille.drop(columns=["lambx", "lamby", "lat_grille", "lon_grille"]),
    left_on="numero_grille_proche", right_on="NUMERO", how="left"
).drop(columns=["NUMERO", "numero_grille_proche"])

print("\nCouverture après rattachement spatial :")
print(f"{(df_communes_swi['swi_moyen'].notna().mean() * 100):.1f}% des communes ont désormais des données SWI")

# 8. Reconstruire la table finale avec ce nouveau rattachement SWI.................................

df_final_v2 = (
    cible_pivot
    .merge(df_communes_swi, on="code_commune", how="left")
)

print("\nDimensions table finale v2 :", df_final_v2.shape)
print("\n--- Valeurs manquantes par colonne ---")
print(df_final_v2.isna().sum())

Points de grille SWI : (8981, 8)
Communes avec géoloc : (38932, 5)

Couverture après rattachement spatial :
100.0% des communes ont désormais des données SWI

Dimensions table finale v2 : (27304, 19)

--- Valeurs manquantes par colonne ---
code_commune                   0
risque_Autres périls           0
risque_Avalanche               0
risque_Inondations             0
risque_Mouvement de terrain    0
risque_Sécheresse              0
risque_Séisme                  0
risque_Tous périls             0
risque_Vent cyclonique         0
latitude                       5
longitude                      5
code_departement               5
nom_departement                5
distance_deg                   5
swi_moyen                      5
swi_variabilite                5
swi_min                        5
swi_max                        5
nb_mois_secheresse             5
dtype: int64


In [ ]:
# Identifier les 5 communes orphelines (pour info / rapport)....................
print(df_final_v2[df_final_v2["latitude"].isna()]["code_commune"].tolist())

# Les retirer (5 sur 27304, impact négligeable).................................
df_final_v2 = df_final_v2.dropna(subset=["latitude"]).reset_index(drop=True)
print("Dimensions finales :", df_final_v2.shape)

['13055', '48166', '69123', '75056', '76095']
Dimensions finales : (27299, 19)


In [ ]:
# Vérifier les catégories de risque dans catnat_staging (table brute avec dates).....................
query_check_libelles = """
SELECT DISTINCT lib_risque_jo, COUNT(*) AS nb_arretes
FROM `sous-nos-pieds.staging.catnat_staging`
GROUP BY lib_risque_jo
ORDER BY nb_arretes DESC
"""
print(client.query(query_check_libelles).to_dataframe())

# Vérifier la plage de dates disponible................................................................
query_check_dates = """
SELECT MIN(date_debut) AS date_min, MAX(date_debut) AS date_max
FROM `sous-nos-pieds.staging.catnat_staging`
"""
print(client.query(query_check_dates).to_dataframe())

                                  lib_risque_jo  nb_arretes
0             inondations et/ou coulees de boue      139419
1                                    secheresse       43417
2                          mouvement de terrain       31711
3                                       tempete       16025
4   chocs mecaniques lies a l'action des vagues        6720
5                         glissement de terrain        3182
6                             poids de la neige        2758
7                    inondations remontee nappe        1368
8                                         grele        1333
9                             secousse sismique         749
10                                    avalanche         119
11               effondrement et/ou affaisement         105
12                            vents cycloniques          85
13                                       divers          65
14              eboulement et/ou chute de blocs          63
15                                 raz d

In [ ]:
import requests
import time


# TEST 1 : Zonage sismique sur une commune connue....................................

url_test = "https://www.georisques.gouv.fr/api/v1/zonage_sismique"
params = {"code_insee": "75056"}  # Paris, à titre de test.... hoping it works ya zah !!!!
response = requests.get(url_test, params=params)
print("Status:", response.status_code)
print(response.json())

Status: 200
{'results': 0, 'page': 1, 'total_pages': 0, 'data': [], 'response_code': 200, 'message': None, 'next': None, 'previous': None}


In [ ]:

# TEST 2 : Aléa argile sur la même commune

url_test_argile = "https://www.georisques.gouv.fr/api/v1/gaspar/argiles"
response_argile = requests.get(url_test_argile, params=params)
print("Status:", response_argile.status_code)
print(response_argile.json())

Status: 404
{'message': 'no Route matched with those values', 'request_id': '2b43112d8d8e496445db170d5766a45d'}


In [ ]:
url_rapport = "https://www.georisques.gouv.fr/api/v1/resultats_rapport_risque"
params = {"code_insee": "75056"}
response_rapport = requests.get(url_rapport, params=params, headers={"Accept": "application/json"})
print("Status:", response_rapport.status_code)
print(response_rapport.json())

Status: 200
{'adresse': None, 'commune': {'libelle': 'PARIS', 'codePostal': '75001', 'codeInsee': '75056'}, 'url': 'https://www.georisques.gouv.fr/mes-risques/connaitre-les-risques-pres-de-chez-moi/rapport2/75056/PARIS/commune/75001', 'risquesNaturels': {'inondation': {'present': True, 'libelle': 'Inondation', 'libelleStatutCommune': 'Risque Existant', 'libelleStatutAdresse': None, 'specifique': None}, 'remonteeNappe': {'present': True, 'libelle': 'Remontée de nappe', 'libelleStatutCommune': 'Risque Existant', 'libelleStatutAdresse': None, 'specifique': None}, 'risqueCotier': {'present': False, 'libelle': 'Risques côtiers (submersion marine, tsunami)', 'libelleStatutCommune': None, 'libelleStatutAdresse': None, 'specifique': None}, 'seisme': {'present': True, 'libelle': 'Séisme', 'libelleStatutCommune': 'Risque Existant - faible', 'libelleStatutAdresse': None, 'specifique': None}, 'mouvementTerrain': {'present': True, 'libelle': 'Mouvements de terrain', 'libelleStatutCommune': 'Risque 

In [ ]:
import json
print(json.dumps(response_rapport.json(), indent=2, ensure_ascii=False))

{
  "adresse": null,
  "commune": {
    "libelle": "PARIS",
    "codePostal": "75001",
    "codeInsee": "75056"
  },
  "url": "https://www.georisques.gouv.fr/mes-risques/connaitre-les-risques-pres-de-chez-moi/rapport2/75056/PARIS/commune/75001",
  "risquesNaturels": {
    "inondation": {
      "present": true,
      "libelle": "Inondation",
      "libelleStatutCommune": "Risque Existant",
      "libelleStatutAdresse": null,
      "specifique": null
    },
    "remonteeNappe": {
      "present": true,
      "libelle": "Remontée de nappe",
      "libelleStatutCommune": "Risque Existant",
      "libelleStatutAdresse": null,
      "specifique": null
    },
    "risqueCotier": {
      "present": false,
      "libelle": "Risques côtiers (submersion marine, tsunami)",
      "libelleStatutCommune": null,
      "libelleStatutAdresse": null,
      "specifique": null
    },
    "seisme": {
      "present": true,
      "libelle": "Séisme",
      "libelleStatutCommune": "Risque Existant - faible",


In [ ]:
import requests
import time
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

def get_risques_commune(code_insee):
    """Appelle l'API Géorisques et extrait argile + séisme pour une commune."""
    url = "https://www.georisques.gouv.fr/api/v1/resultats_rapport_risque"
    try:
        r = requests.get(url, params={"code_insee": code_insee},
                          headers={"Accept": "application/json"}, timeout=10)
        if r.status_code != 200:
            return {"code_commune": code_insee, "argile_statut": None, "seisme_statut": None, "erreur": r.status_code}
        data = r.json()
        risques = data.get("risquesNaturels", {})
        argile = risques.get("retraitGonflementArgile", {})
        seisme = risques.get("seisme", {})
        return {
            "code_commune": code_insee,
            "argile_present": argile.get("present"),
            "argile_statut": argile.get("libelleStatutCommune"),
            "seisme_present": seisme.get("present"),
            "seisme_statut": seisme.get("libelleStatutCommune"),
            "erreur": None
        }
    except Exception as e:
        return {"code_commune": code_insee, "argile_statut": None, "seisme_statut": None, "erreur": str(e)}


# Test sur un petit échantillon d'abord (50 communes)

echantillon = df_final_v2["code_commune"].head(50).tolist()
resultats_test = [get_risques_commune(c) for c in echantillon]
df_test = pd.DataFrame(resultats_test)
print(df_test["erreur"].value_counts(dropna=False))
print(df_test.head(10))

erreur
None                                                                              37
('Connection aborted.', ConnectionResetError(104, 'Connection reset by peer'))    12
404                                                                                1
Name: count, dtype: int64
  code_commune argile_present                argile_statut seisme_present  \
0        01001           True     Risque Existant - modéré           True   
1        01002           True  Risque Existant - important           True   
2        01004            NaN                         None            NaN   
3        01005           True     Risque Existant - modéré           True   
4        01006           True  Risque Existant - important           True   
5        01007           True  Risque Existant - important           True   
6        01008           True  Risque Existant - important           True   
7        01009           True  Risque Existant - important           True   
8        01011     

In [ ]:
import requests
import time

# TEST 1 : Zonage sismique sur une commune connue.................

url_test = "https://www.georisques.gouv.fr/api/v1/zonage_sismique"
params = {"code_insee": "75056"}  # Paris, à titre de test
response = requests.get(url_test, params=params)
print("Status:", response.status_code)
print(response.json())

Status: 200
{'results': 0, 'page': 1, 'total_pages': 0, 'data': [], 'response_code': 200, 'message': None, 'next': None, 'previous': None}


In [ ]:

# TEST 2 : Aléa argile sur la même commune..............................

url_test_argile = "https://www.georisques.gouv.fr/api/v1/gaspar/argiles"
response_argile = requests.get(url_test_argile, params=params)
print("Status:", response_argile.status_code)
print(response_argile.json())

Status: 404
{'message': 'no Route matched with those values', 'request_id': 'f93a07036a311139ecfade3161788152'}


In [ ]:
#why why whyyyyyyyyy

url_rapport = "https://www.georisques.gouv.fr/api/v1/resultats_rapport_risque"
params = {"code_insee": "75056"}
response_rapport = requests.get(url_rapport, params=params, headers={"Accept": "application/json"})
print("Status:", response_rapport.status_code)
print(response_rapport.json())

Status: 200
{'adresse': None, 'commune': {'libelle': 'PARIS', 'codePostal': '75001', 'codeInsee': '75056'}, 'url': 'https://www.georisques.gouv.fr/mes-risques/connaitre-les-risques-pres-de-chez-moi/rapport2/75056/PARIS/commune/75001', 'risquesNaturels': {'inondation': {'present': True, 'libelle': 'Inondation', 'libelleStatutCommune': 'Risque Existant', 'libelleStatutAdresse': None, 'specifique': None}, 'remonteeNappe': {'present': True, 'libelle': 'Remontée de nappe', 'libelleStatutCommune': 'Risque Existant', 'libelleStatutAdresse': None, 'specifique': None}, 'risqueCotier': {'present': False, 'libelle': 'Risques côtiers (submersion marine, tsunami)', 'libelleStatutCommune': None, 'libelleStatutAdresse': None, 'specifique': None}, 'seisme': {'present': True, 'libelle': 'Séisme', 'libelleStatutCommune': 'Risque Existant - faible', 'libelleStatutAdresse': None, 'specifique': None}, 'mouvementTerrain': {'present': True, 'libelle': 'Mouvements de terrain', 'libelleStatutCommune': 'Risque 

In [ ]:
import json
print(json.dumps(response_rapport.json(), indent=2, ensure_ascii=False))

{
  "adresse": null,
  "commune": {
    "libelle": "PARIS",
    "codePostal": "75001",
    "codeInsee": "75056"
  },
  "url": "https://www.georisques.gouv.fr/mes-risques/connaitre-les-risques-pres-de-chez-moi/rapport2/75056/PARIS/commune/75001",
  "risquesNaturels": {
    "inondation": {
      "present": true,
      "libelle": "Inondation",
      "libelleStatutCommune": "Risque Existant",
      "libelleStatutAdresse": null,
      "specifique": null
    },
    "remonteeNappe": {
      "present": true,
      "libelle": "Remontée de nappe",
      "libelleStatutCommune": "Risque Existant",
      "libelleStatutAdresse": null,
      "specifique": null
    },
    "risqueCotier": {
      "present": false,
      "libelle": "Risques côtiers (submersion marine, tsunami)",
      "libelleStatutCommune": null,
      "libelleStatutAdresse": null,
      "specifique": null
    },
    "seisme": {
      "present": true,
      "libelle": "Séisme",
      "libelleStatutCommune": "Risque Existant - faible",


In [ ]:
import requests
import time
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

def get_risques_commune(code_insee):
    """Appelle l'API Géorisques et extrait argile + séisme pour une commune."""
    url = "https://www.georisques.gouv.fr/api/v1/resultats_rapport_risque"
    try:
        r = requests.get(url, params={"code_insee": code_insee},
                          headers={"Accept": "application/json"}, timeout=10)
        if r.status_code != 200:
            return {"code_commune": code_insee, "argile_statut": None, "seisme_statut": None, "erreur": r.status_code}
        data = r.json()
        risques = data.get("risquesNaturels", {})
        argile = risques.get("retraitGonflementArgile", {})
        seisme = risques.get("seisme", {})
        return {
            "code_commune": code_insee,
            "argile_present": argile.get("present"),
            "argile_statut": argile.get("libelleStatutCommune"),
            "seisme_present": seisme.get("present"),
            "seisme_statut": seisme.get("libelleStatutCommune"),
            "erreur": None
        }
    except Exception as e:
        return {"code_commune": code_insee, "argile_statut": None, "seisme_statut": None, "erreur": str(e)}

# Test sur un petit échantillon d'abord (50 communes)...........................

echantillon = df_final_v2["code_commune"].head(50).tolist()
resultats_test = [get_risques_commune(c) for c in echantillon]
df_test = pd.DataFrame(resultats_test)
print(df_test["erreur"].value_counts(dropna=False))
print(df_test.head(10))

erreur
None                                                                              26
('Connection aborted.', ConnectionResetError(104, 'Connection reset by peer'))    24
Name: count, dtype: int64
  code_commune                argile_statut             seisme_statut  \
0        01001                         None                      None   
1        01002  Risque Existant - important  Risque Existant - modéré   
2        01004  Risque Existant - important  Risque Existant - modéré   
3        01005     Risque Existant - modéré  Risque Existant - faible   
4        01006  Risque Existant - important  Risque Existant - modéré   
5        01007                         None                      None   
6        01008  Risque Existant - important  Risque Existant - modéré   
7        01009  Risque Existant - important  Risque Existant - modéré   
8        01011  Risque Existant - important  Risque Existant - modéré   
9        01014  Risque Existant - important  Risque Existant - modé

In [ ]:

# Reconstruire la base sur TOUTES les communes françaises......:/


# Base complète = référentiel géo (déjà normalisé sur 5 caractères)................
base_communes = df_geo[["code_commune"]].drop_duplicates()
print("Nombre de communes de référence :", base_communes.shape[0])

# LEFT JOIN cible depuis cette base complète (au lieu de partir de cible_pivot)..............
df_cible_complet = base_communes.merge(cible_pivot, on="code_commune", how="left")

# Les communes sans aucune déclaration catnat -> 0 sur toutes les colonnes risque_*.................
colonnes_risque = [c for c in df_cible_complet.columns if c.startswith("risque_")]
df_cible_complet[colonnes_risque] = df_cible_complet[colonnes_risque].fillna(0).astype(int)

print("Répartition risque_Inondations avant/après correction :")
print(df_cible_complet["risque_Inondations"].value_counts())

Nombre de communes de référence : 36013
Répartition risque_Inondations avant/après correction :
risque_Inondations
1    23490
0    12523
Name: count, dtype: int64


In [ ]:
df_final_v3 = (
    df_cible_complet
    .merge(df_communes_swi.drop(columns=["latitude", "longitude"]), on="code_commune", how="left")
)

print("Dimensions table finale v3 :", df_final_v3.shape)
print(df_final_v3.isna().sum())

Dimensions table finale v3 : (39024, 17)
code_commune                    0
risque_Autres périls            0
risque_Avalanche                0
risque_Inondations              0
risque_Mouvement de terrain     0
risque_Sécheresse               0
risque_Séisme                   0
risque_Tous périls              0
risque_Vent cyclonique          0
code_departement               92
nom_departement                92
distance_deg                   92
swi_moyen                      92
swi_variabilite                92
swi_min                        92
swi_max                        92
nb_mois_secheresse             92
dtype: int64


In [ ]:
# Vérifier les doublons dans df_communes_swi..................
print("Doublons dans df_communes_swi :", df_communes_swi["code_commune"].duplicated().sum())

# Vérifier les doublons dans df_final_v3.........................
print("Doublons dans df_final_v3 :", df_final_v3["code_commune"].duplicated().sum())

# Si doublons confirmés, les examiner.....................................;
if df_communes_swi["code_commune"].duplicated().sum() > 0:
    doublons = df_communes_swi[df_communes_swi["code_commune"].duplicated(keep=False)].sort_values("code_commune")
    print(doublons.head(10))

Doublons dans df_communes_swi : 3011
Doublons dans df_final_v3 : 3011
   code_commune   latitude  longitude code_departement nom_departement  \
13        01015  45.723762   5.652631                1             ain   
14        01015  45.723762   5.652631                1             ain   
29        01032  45.859146   5.139895                1             ain   
30        01032  45.859146   5.139895                1             ain   
31        01032  45.859146   5.139895                1             ain   
32        01032  45.859146   5.139895                1             ain   
36        01036  45.884748   5.651158                1             ain   
37        01036  45.884748   5.651158                1             ain   
50        01050  46.372797   4.998976                1             ain   
51        01050  46.372797   4.998976                1             ain   

    distance_deg  swi_moyen  swi_variabilite  swi_min  swi_max  \
13      0.038238   0.752333         0.292360    0

In [ ]:
# Dédoublonner en gardant une seule ligne par commune...........................
df_communes_swi_dedupe = df_communes_swi.drop_duplicates(subset="code_commune", keep="first")
print("Avant dédoublonnage :", df_communes_swi.shape[0])
print("Après dédoublonnage :", df_communes_swi_dedupe.shape[0])

# Refaire la jointure finale avec la version dédupliquée.......................
df_final_v3 = (
    df_cible_complet
    .merge(df_communes_swi_dedupe.drop(columns=["latitude", "longitude"]), on="code_commune", how="left")
)

print("\nDimensions table finale v3 (corrigée) :", df_final_v3.shape)
print(df_final_v3.isna().sum())
print("\nDoublons restants :", df_final_v3["code_commune"].duplicated().sum())

Avant dédoublonnage : 38932
Après dédoublonnage : 35921

Dimensions table finale v3 (corrigée) : (36013, 17)
code_commune                    0
risque_Autres périls            0
risque_Avalanche                0
risque_Inondations              0
risque_Mouvement de terrain     0
risque_Sécheresse               0
risque_Séisme                   0
risque_Tous périls              0
risque_Vent cyclonique          0
code_departement               92
nom_departement                92
distance_deg                   92
swi_moyen                      92
swi_variabilite                92
swi_min                        92
swi_max                        92
nb_mois_secheresse             92
dtype: int64

Doublons restants : 0


In [ ]:
tous_les_codes = df_final_v3["code_commune"].tolist()
print(f"Nombre de communes à traiter : {len(tous_les_codes)}")

resultats = []
taille_lot = 500

with ThreadPoolExecutor(max_workers=10) as executor:
    futures = {executor.submit(get_risques_commune, code): code for code in tous_les_codes}
    for i, future in enumerate(as_completed(futures), 1):
        resultats.append(future.result())
        if i % taille_lot == 0 or i == len(tous_les_codes):
            df_partiel = pd.DataFrame(resultats)
            df_partiel.to_csv("/content/risques_argile_seisme_progress.csv", index=False)
            print(f"{i}/{len(tous_les_codes)} traités — {df_partiel['erreur'].isna().sum()} succès")

df_risques_geo = pd.DataFrame(resultats)
print("\nTerminé.")
print(df_risques_geo["erreur"].value_counts(dropna=False))

Nombre de communes à traiter : 36013
500/36013 traités — 132 succès
1000/36013 traités — 132 succès
1500/36013 traités — 132 succès
2000/36013 traités — 132 succès
2500/36013 traités — 132 succès
3000/36013 traités — 132 succès
3500/36013 traités — 132 succès
4000/36013 traités — 132 succès
4500/36013 traités — 132 succès
5000/36013 traités — 235 succès
5500/36013 traités — 235 succès
6000/36013 traités — 235 succès
6500/36013 traités — 235 succès
7000/36013 traités — 237 succès
7500/36013 traités — 239 succès
8000/36013 traités — 239 succès
8500/36013 traités — 239 succès
9000/36013 traités — 239 succès
9500/36013 traités — 239 succès
10000/36013 traités — 239 succès
10500/36013 traités — 239 succès
11000/36013 traités — 239 succès
11500/36013 traités — 239 succès
12000/36013 traités — 239 succès
12500/36013 traités — 239 succès
13000/36013 traités — 239 succès
13500/36013 traités — 239 succès
14000/36013 traités — 239 succès
14500/36013 traités — 239 succès
15000/36013 traités — 239 

In [ ]:
df_diag = pd.DataFrame(resultats)
print(df_diag["erreur"].value_counts(dropna=False).head(20))

erreur
('Connection aborted.', ConnectionResetError(104, 'Connection reset by peer'))    35612
None                                                                                376
404                                                                                  25
Name: count, dtype: int64


In [ ]:
import requests, time
from concurrent.futures import ThreadPoolExecutor, as_completed

def get_risques_commune(code_insee, max_retries=3):
    url = "https://www.georisques.gouv.fr/api/v1/resultats_rapport_risque"
    for tentative in range(max_retries):
        try:
            r = requests.get(url, params={"code_insee": code_insee},
                              headers={"Accept": "application/json"}, timeout=15)
            if r.status_code != 200:
                return {"code_commune": code_insee, "argile_statut": None, "seisme_statut": None, "erreur": r.status_code}
            data = r.json()
            risques = data.get("risquesNaturels", {})
            argile = risques.get("retraitGonflementArgile", {})
            seisme = risques.get("seisme", {})
            return {
                "code_commune": code_insee,
                "argile_present": argile.get("present"),
                "argile_statut": argile.get("libelleStatutCommune"),
                "seisme_present": seisme.get("present"),
                "seisme_statut": seisme.get("libelleStatutCommune"),
                "erreur": None
            }
        except Exception as e:
            if tentative < max_retries - 1:
                time.sleep(2 ** tentative)  # backoff exponentiel : 1s, 2s, 4s...
                continue
            return {"code_commune": code_insee, "argile_statut": None, "seisme_statut": None, "erreur": str(e)}

# Réduire drastiquement le parallélisme et ajouter une pause entre lots
with ThreadPoolExecutor(max_workers=3) as executor:  # 3 au lieu de 10
    futures = {executor.submit(get_risques_commune, code): code for code in tous_les_codes}
    for i, future in enumerate(as_completed(futures), 1):
        resultats.append(future.result())
        if i % 100 == 0:
            time.sleep(1)  # petite pause pour ne pas marteler l'API

In [ ]:
!pip install geopandas --quiet

import geopandas as gpd

# Lecture du shapefile argile........................................................
gdf_argile = gpd.read_file("/content/AleaRG_2025_Fxx_L93.shp")
print("CRS :", gdf_argile.crs)
print("Nombre de polygones :", len(gdf_argile))
print("\nColonnes disponibles :")
print(gdf_argile.columns.tolist())
print("\nAperçu :")
print(gdf_argile.head())

In [ ]:
import geopandas as gpd
from shapely.geometry import Point

# Créer un GeoDataFrame des communes à partir de leurs lat/long (WGS84 -> reprojeter en Lambert-93 pour matcher l'argile).......................
gdf_communes = gpd.GeoDataFrame(
    df_communes.copy(),
    geometry=gpd.points_from_xy(df_communes["longitude"], df_communes["latitude"]),
    crs="EPSG:4326"
).to_crs("EPSG:2154")  # reprojection pour matcher le CRS du shapefile argile..........................................

print("Communes à géolocaliser :", gdf_communes.shape[0])

# Jointure spatiale : quel polygone argile contient chaque point commune ?..........................................
gdf_argile_join = gpd.sjoin(
    gdf_communes[["code_commune", "geometry"]],
    gdf_argile[["niveau", "geometry"]],
    how="left",
    predicate="within"
)

print("\nRésultat jointure :")
print(gdf_argile_join["niveau"].value_counts(dropna=False))
print("\nTaux de couverture :", (gdf_argile_join["niveau"].notna().mean() * 100), "%")

# Nettoyer : une commune peut matcher 2 polygones adjacents (bordure) -> garder une seule ligne par commune........................
df_argile_par_commune = (
    gdf_argile_join.groupby("code_commune")["niveau"]
    .max()  # en cas de doublon bordure, on garde le niveau de risque le plus élevé (le plus prudent)...............................
    .reset_index()
    .rename(columns={"niveau": "argile_niveau"})
)

print("\nCommunes uniques avec niveau argile :", df_argile_par_commune.shape[0])

In [ ]:
print("Bounding box communes (après reprojection EPSG:2154) :")
print(gdf_communes.total_bounds)

print("\nBounding box polygones argile :")
print(gdf_argile.total_bounds)

In [ ]:
# 1. Isoler uniquement les communes de métropole (départements 01-95, 2A, 2B)..............................
depts_metropole = [f"{i:02d}" for i in range(1, 96)] + ["2A", "2B"]
gdf_communes_metro = gdf_communes[gdf_communes["code_commune"].str[:2].isin(depts_metropole)].copy()

print("Communes métropole isolées :", gdf_communes_metro.shape[0])
print("Bounding box métropole (après reprojection) :")
print(gdf_communes_metro.total_bounds)

# 2. Vérifier la validité des géométries argile (polygones potentiellement invalides).......................
print("\nPolygones argile valides :", gdf_argile.geometry.is_valid.sum(), "/", len(gdf_argile))

# 3. Refaire la jointure sur les seules communes métropole............................
gdf_argile_join_metro = gpd.sjoin(
    gdf_communes_metro[["code_commune", "geometry"]],
    gdf_argile[["niveau", "geometry"]],
    how="left",
    predicate="within"
)
print("\nTaux de couverture (métropole uniquement) :",
      (gdf_argile_join_metro["niveau"].notna().mean() * 100), "%")

In [ ]:
# Réparer les géométries invalides (technique standard : buffer(0))......................
gdf_argile["geometry"] = gdf_argile.geometry.buffer(0)

print("Polygones valides après réparation :", gdf_argile.geometry.is_valid.sum(), "/", len(gdf_argile))

# Refaire la jointure spatiale sur les communes métropole avec les géométries réparées.......................
gdf_argile_join_metro = gpd.sjoin(
    gdf_communes_metro[["code_commune", "geometry"]],
    gdf_argile[["niveau", "geometry"]],
    how="left",
    predicate="within"
)
print("\nTaux de couverture (métropole, géométries réparées) :",
      (gdf_argile_join_metro["niveau"].notna().mean() * 100), "%")
print(gdf_argile_join_metro["niveau"].value_counts(dropna=False))

In [ ]:
# Surface totale des polygones argile (en m², somme de la colonne surf_m2)........................................
surface_argile_km2 = gdf_argile["surf_m2"].sum() / 1e6
print(f"Surface totale couverte par la carte argile : {surface_argile_km2:,.0f} km²")

# Surface de la France métropolitaine ≈ 543 940 km²..............................................
surface_france_km2 = 543940
print(f"Proportion du territoire couverte : {(surface_argile_km2 / surface_france_km2 * 100):.1f}%")

In [ ]:
# Jointure par plus proche voisin (tolère les micro-écarts entre polygones adjacents)...................................
gdf_argile_nearest = gpd.sjoin_nearest(
    gdf_communes_metro[["code_commune", "geometry"]],
    gdf_argile[["niveau", "geometry"]],
    how="left",
    max_distance=1000,  # tolérance de 1km,  pour des écarts topologiques.................................
    distance_col="dist_argile"
)

print("Taux de couverture (sjoin_nearest) :", (gdf_argile_nearest["niveau"].notna().mean() * 100), "%")
print(gdf_argile_nearest["niveau"].value_counts(dropna=False))
print("\nDistance moyenne au polygone le plus proche (m) :", gdf_argile_nearest["dist_argile"].mean())

In [ ]:
gdf_argile_nearest = gpd.sjoin_nearest(
    gdf_communes_metro[["code_commune", "geometry"]],
    gdf_argile[["niveau", "geometry"]],
    how="left",
    max_distance=15000,  # 15km, pour capturer les centres urbains non cartographiés...............................
    distance_col="dist_argile"
)

print("Taux de couverture (max_distance=15km) :", (gdf_argile_nearest["niveau"].notna().mean() * 100), "%")
print(gdf_argile_nearest["niveau"].value_counts(dropna=False))
print("\nDistance moyenne (m) :", gdf_argile_nearest["dist_argile"].mean())
print("Distance médiane (m) :", gdf_argile_nearest["dist_argile"].median())
print("Distance max (m) :", gdf_argile_nearest["dist_argile"].max())

In [ ]:
gdf_argile_nearest = gpd.sjoin_nearest(
    gdf_communes_metro[["code_commune", "geometry"]],
    gdf_argile[["niveau", "geometry"]],
    how="left",
    max_distance=15000,  # 15km pour capturer les centres urbains non cartographiés...............................................
    distance_col="dist_argile"
)

print("Taux de couverture (max_distance=15km) :", (gdf_argile_nearest["niveau"].notna().mean() * 100), "%")
print(gdf_argile_nearest["niveau"].value_counts(dropna=False))
print("\nDistance moyenne (m) :", gdf_argile_nearest["dist_argile"].mean())
print("Distance médiane (m) :", gdf_argile_nearest["dist_argile"].median())
print("Distance max (m) :", gdf_argile_nearest["dist_argile"].max())

In [ ]:
# Finaliser la table argile par commune (garder le niveau le plus élevé en cas de doublon)
df_argile_par_commune = (
    gdf_argile_nearest.groupby("code_commune")["niveau"]
    .max()
    .reset_index()
    .rename(columns={"niveau": "argile_niveau"})
)

# Réintégrer toutes les communes métropole (pour ne perdre aucune commune)
df_argile_par_commune = (
    gdf_communes_metro[["code_commune"]]
    .merge(df_argile_par_commune, on="code_commune", how="left")
)

print("Total communes :", df_argile_par_commune.shape[0])
print("NaN restants :", df_argile_par_commune["argile_niveau"].isna().sum())
print(df_argile_par_commune["argile_niveau"].value_counts(dropna=False))

In [ ]:
import requests
import zipfile
import os

# Téléchargement du zonage sismique.................... API lets go

url_sismique = "https://www.data.gouv.fr/api/1/datasets/r/4f6bc1ec-5e29-404e-8214-bef23088eaf6"
r = requests.get(url_sismique, timeout=120)
with open("/content/zonage_sismique.zip", "wb") as f:
    f.write(r.content)
print("Téléchargé :", os.path.getsize("/content/zonage_sismique.zip") / 1e6, "Mo")

with zipfile.ZipFile("/content/zonage_sismique.zip", "r") as z:
    z.extractall("/content/zonage_sismique/")
    print(z.namelist())

In [ ]:
import geopandas as gpd

gdf_sismique = gpd.read_file("/content/zonage_sismique/France_zonage_sismique.shp")
print("CRS :", gdf_sismique.crs)
print("Nombre d'entités :", len(gdf_sismique))
print("\nColonnes disponibles :")
print(gdf_sismique.columns.tolist())
print("\nAperçu :")
print(gdf_sismique.head())

In [ ]:
import pandas as pd

# Extraction directe (pas de jointure spatiale nécessaire)..............................;;


df_sismique = gdf_sismique[["insee", "Sismicite"]].copy()

# Normaliser le code commune sur 5 caractères............................
df_sismique["code_commune"] = df_sismique["insee"].astype(str).str.zfill(5)

# Extraire le niveau numérique depuis "4 - Moyenne" -> 4
df_sismique["sismique_niveau"] = df_sismique["Sismicite"].str.extract(r"^(\d+)").astype(float)

# Vérifier les doublons avant de merger..........................................
print("Doublons code_commune :", df_sismique["code_commune"].duplicated().sum())

# Dédoublonner si besoin (garder le niveau max par sécurité)...................................
df_sismique_par_commune = (
    df_sismique.groupby("code_commune")["sismique_niveau"]
    .max()
    .reset_index()
)

print("\nCommunes uniques avec niveau sismique :", df_sismique_par_commune.shape[0])
print(df_sismique_par_commune["sismique_niveau"].value_counts().sort_index())

In [ ]:

# Assemblage final : df_final_v3 + argile + sismique.............................................

df_final_complet = (
    df_final_v3
    .merge(df_argile_par_commune, on="code_commune", how="left")
    .merge(df_sismique_par_commune, on="code_commune", how="left")
)

print("Dimensions table finale complète :", df_final_complet.shape)
print("\n--- Valeurs manquantes par colonne ---")
print(df_final_complet.isna().sum())

In [ ]:
print("Doublons argile :", df_argile_par_commune["code_commune"].duplicated().sum())
print("Doublons sismique :", df_sismique_par_commune["code_commune"].duplicated().sum())

In [ ]:
df_argile_par_commune_clean = df_argile_par_commune.drop_duplicates(subset="code_commune", keep="first")
print("Avant :", df_argile_par_commune.shape[0], "-> Après :", df_argile_par_commune_clean.shape[0])

# Refaire l'assemblage final avec la version nettoyée...............................................
df_final_complet = (
    df_final_v3
    .merge(df_argile_par_commune_clean, on="code_commune", how="left")
    .merge(df_sismique_par_commune, on="code_commune", how="left")
)

print("\nDimensions table finale (corrigée) :", df_final_complet.shape)
print(df_final_complet.isna().sum())
print("\nDoublons finaux :", df_final_complet["code_commune"].duplicated().sum())

In [ ]:

# Imputation des NaN restants par médiane départementale..............................................

colonnes_a_imputer = ["swi_moyen", "swi_variabilite", "swi_min", "swi_max",
                       "nb_mois_secheresse", "argile_niveau", "sismique_niveau"]

for col in colonnes_a_imputer:
    df_final_complet[col] = df_final_complet.groupby("code_departement")[col].transform(
        lambda x: x.fillna(x.median())
    )

# S'il reste des NaN (département entier sans aucune valeur), fallback sur médiane globale.............................
for col in colonnes_a_imputer:
    df_final_complet[col] = df_final_complet[col].fillna(df_final_complet[col].median())

print("NaN restants après imputation :")
print(df_final_complet[colonnes_a_imputer].isna().sum())
print("\nDimensions finales :", df_final_complet.shape)

In [ ]:

# Reconstruction de la cible avec dates (pour split temporel).............................................................

query_cible_temporelle = """
SELECT
  LPAD(CAST(code_commune AS STRING), 5, '0') AS code_commune,
  lib_risque_jo,
  EXTRACT(YEAR FROM date_debut) AS annee
FROM `sous-nos-pieds.staging.catnat_staging`
WHERE date_debut IS NOT NULL
"""
df_cible_temporelle = client.query(query_cible_temporelle).to_dataframe()

print("Dimensions :", df_cible_temporelle.shape)
print("\nPlage d'années :", df_cible_temporelle["annee"].min(), "-", df_cible_temporelle["annee"].max())
print("\nLibellés de risque distincts :")
print(df_cible_temporelle["lib_risque_jo"].value_counts())

In [ ]:
mapping_risque = {
    "inondations et/ou coulees de boue": "Inondations",
    "inondations remontee nappe": "Inondations",
    "coulee de boue": "Inondations",
    "secheresse": "Sécheresse",
    "mouvement de terrain": "Mouvement de terrain",
    "glissement de terrain": "Mouvement de terrain",
    "effondrement et/ou affaisement": "Mouvement de terrain",
    "eboulement et/ou chute de blocs": "Mouvement de terrain",
    "lave torrentielle": "Mouvement de terrain",
    "glissement et effondrement de terrain": "Mouvement de terrain",
    "glissement et eboulement rocheux": "Mouvement de terrain",
    "secousse sismique": "Séisme",
    "avalanche": "Avalanche",
    "vents cycloniques": "Vent cyclonique",
    "tempete": "Autres périls",
    "chocs mecaniques lies a l'action des vagues": "Autres périls",
    "poids de la neige": "Autres périls",
    "grele": "Autres périls",
    "divers": "Autres périls",
    "raz de maree": "Autres périls",
    "eruption volcanique": "Autres périls",
}

df_cible_temporelle["risque_categorie"] = df_cible_temporelle["lib_risque_jo"].map(mapping_risque)

# Vérifier qu'aucun libellé n'a été oublié..............again..............................
print("Libellés non mappés :", df_cible_temporelle[df_cible_temporelle["risque_categorie"].isna()]["lib_risque_jo"].unique())

In [ ]:

# Split temporel : train (avant 2019) / test (2019+).......................................
: train (avant 2019) / test (2019+)

ANNEE_COUPURE = 2019

def construire_cible(df, annees_filtre):
    df_filtre = df[df["annee"].isin(annees_filtre)]
    pivot = (
        df_filtre.pivot_table(index="code_commune", columns="risque_categorie",
                               aggfunc="size", fill_value=0)
        .gt(0).astype(int)
        .add_prefix("risque_")
        .reset_index()
    )
    return pivot

annees_train = range(df_cible_temporelle["annee"].min(), ANNEE_COUPURE)
annees_test = range(ANNEE_COUPURE, df_cible_temporelle["annee"].max() + 1)

cible_train = construire_cible(df_cible_temporelle, annees_train)
cible_test = construire_cible(df_cible_temporelle, annees_test)

print("Cible train :", cible_train.shape, "| Cible test :", cible_test.shape)

In [ ]:

# Assembler features + cibles temporelles (train et test séparément).....................................

colonnes_features = ["code_commune", "code_departement", "nom_departement",
                      "swi_moyen", "swi_variabilite", "swi_min", "swi_max",
                      "nb_mois_secheresse", "argile_niveau", "sismique_niveau"]

df_features = df_final_complet[colonnes_features].copy()

colonnes_risque = ["risque_Autres périls", "risque_Avalanche", "risque_Inondations",
                    "risque_Mouvement de terrain", "risque_Sécheresse",
                    "risque_Séisme", "risque_Vent cyclonique"]

def assembler(df_features, cible, colonnes_risque):
    df_assemble = df_features.merge(cible, on="code_commune", how="left")
    for col in colonnes_risque:
        if col not in df_assemble.columns:
            df_assemble[col] = 0
        df_assemble[col] = df_assemble[col].fillna(0).astype(int)
    return df_assemble

df_train = assembler(df_features, cible_train, colonnes_risque)
df_test = assembler(df_features, cible_test, colonnes_risque)

print("Train :", df_train.shape, "| Test :", df_test.shape)
print("\nRépartition risque_Inondations (train) :")
print(df_train["risque_Inondations"].value_counts())
print("\nRépartition risque_Inondations (test) :")
print(df_test["risque_Inondations"].value_counts())

In [ ]:
from sklearn.multioutput import MultiOutputClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# Préparation X / y..........................................................................

features_numeriques = ["swi_moyen", "swi_variabilite", "swi_min", "swi_max",
                        "nb_mois_secheresse", "argile_niveau", "sismique_niveau"]

X_train = df_train[features_numeriques]
y_train = df_train[colonnes_risque]

X_test = df_test[features_numeriques]
y_test = df_test[colonnes_risque]

print("X_train :", X_train.shape, "| y_train :", y_train.shape)
print("X_test :", X_test.shape, "| y_test :", y_test.shape)


# Entraînement Random Forest multi-label.......................................................

modele = MultiOutputClassifier(
    RandomForestClassifier(n_estimators=200, class_weight="balanced",
                            max_depth=12, random_state=42, n_jobs=-1)
)
modele.fit(X_train, y_train)

# Évaluation par aléa.............................................................................

y_pred = modele.predict(X_test)

for i, col in enumerate(colonnes_risque):
    print(f"\n=== {col} ===")
    print(classification_report(y_test[col], y_pred[:, i], zero_division=0))

In [ ]:
# Vérifier les probabilités prédites pour Séisme (au lieu du label binaire à seuil 0.5)..................................
idx_seisme = colonnes_risque.index("risque_Séisme")
proba_seisme = modele.estimators_[idx_seisme].predict_proba(X_test)[:, 1]

print("Distribution des probabilités prédites (classe positive) :")
print(pd.Series(proba_seisme).describe())
print("\nNombre de communes avec proba > 0.3 :", (proba_seisme > 0.3).sum())
print("Nombre de communes avec proba > 0.1 :", (proba_seisme > 0.1).sum())

# Importance des features pour ce sous-modèle............................................................................
importances = pd.Series(
    modele.estimators_[idx_seisme].feature_importances_,
    index=features_numeriques
).sort_values(ascending=False)
print("\nImportance des features (Séisme) :")
print(importances)

In [ ]:
from sklearn.metrics import f1_score
import numpy as np

def meilleur_seuil(y_true, y_proba):
    """Trouve le seuil qui maximise le F1-score sur la classe positive."""
    seuils = np.arange(0.05, 0.95, 0.05)
    scores = [f1_score(y_true, (y_proba > s).astype(int), zero_division=0) for s in seuils]
    meilleur_idx = np.argmax(scores)
    return seuils[meilleur_idx], scores[meilleur_idx]

print("=== Recherche du meilleur seuil par aléa ===\n")
seuils_optimaux = {}

for i, col in enumerate(colonnes_risque):
    proba = modele.estimators_[i].predict_proba(X_test)[:, 1]
    seuil, f1 = meilleur_seuil(y_test[col], proba)
    seuils_optimaux[col] = seuil
    print(f"{col:30s} | seuil optimal: {seuil:.2f} | F1: {f1:.3f}")

In [ ]:
print("\n=== Rapports avec seuils optimisés ===")
for i, col in enumerate(colonnes_risque):
    proba = modele.estimators_[i].predict_proba(X_test)[:, 1]
    y_pred_optimise = (proba > seuils_optimaux[col]).astype(int)
    print(f"\n=== {col} (seuil={seuils_optimaux[col]:.2f}) ===")
    print(classification_report(y_test[col], y_pred_optimise, zero_division=0))

In [ ]:

# Regroupement des aléas rares en une seule catégorie................................................................

aleas_rares = ["risque_Séisme", "risque_Avalanche", "risque_Autres périls", "risque_Mouvement de terrain"]
aleas_fiables = ["risque_Inondations", "risque_Sécheresse", "risque_Vent cyclonique"]

df_train["risque_Geophysique_rare"] = (df_train[aleas_rares].sum(axis=1) > 0).astype(int)
df_test["risque_Geophysique_rare"] = (df_test[aleas_rares].sum(axis=1) > 0).astype(int)

colonnes_risque_v2 = aleas_fiables + ["risque_Geophysique_rare"]

print("Nouvelle répartition risque_Geophysique_rare (train) :")
print(df_train["risque_Geophysique_rare"].value_counts())
print("\nNouvelle répartition risque_Geophysique_rare (test) :")
print(df_test["risque_Geophysique_rare"].value_counts())

# Réentraîner sur les 4 catégories (3 fiables + 1 groupée)...................................................................
y_train_v2 = df_train[colonnes_risque_v2]
y_test_v2 = df_test[colonnes_risque_v2]

modele_v2 = MultiOutputClassifier(
    RandomForestClassifier(n_estimators=200, class_weight="balanced",
                            max_depth=12, random_state=42, n_jobs=-1)
)
modele_v2.fit(X_train, y_train_v2)

y_pred_v2 = modele_v2.predict(X_test)
for i, col in enumerate(colonnes_risque_v2):
    print(f"\n=== {col} ===")
    print(classification_report(y_test_v2[col], y_pred_v2[:, i], zero_division=0))

In [ ]:
from sklearn.metrics import f1_score

# Baseline : uniquement les features SWI, sans argile ni sismique.....................................................................
features_baseline = ["swi_moyen", "swi_variabilite", "swi_min", "swi_max", "nb_mois_secheresse"]

X_train_baseline = df_train[features_baseline]
X_test_baseline = df_test[features_baseline]

modele_baseline = MultiOutputClassifier(
    RandomForestClassifier(n_estimators=200, class_weight="balanced",
                            max_depth=12, random_state=42, n_jobs=-1)
)
modele_baseline.fit(X_train_baseline, y_train)

y_pred_baseline = modele_baseline.predict(X_test_baseline)

# Tableau comparatif final : baseline vs enrichi.........................................................................

comparaison = []
for i, col in enumerate(colonnes_risque):
    f1_baseline = f1_score(y_test[col], y_pred_baseline[:, i], zero_division=0)
    proba_enrichi = modele.estimators_[i].predict_proba(X_test)[:, 1]
    y_pred_enrichi_seuil = (proba_enrichi > seuils_optimaux[col]).astype(int)
    f1_enrichi = f1_score(y_test[col], y_pred_enrichi_seuil, zero_division=0)
    comparaison.append({
        "aléa": col.replace("risque_", ""),
        "F1 baseline (SWI seul)": round(f1_baseline, 3),
        "F1 enrichi (+ argile/sismique)": round(f1_enrichi, 3),
        "gain": round(f1_enrichi - f1_baseline, 3)
    })

df_comparaison = pd.DataFrame(comparaison)
print(df_comparaison)

In [ ]:

# Prédictions (probabilités) sur TOUTES les communes, agrégées par département...........................................................

proba_toutes = {}
for i, col in enumerate(colonnes_risque):
    proba_toutes[col.replace("risque_", "proba_")] = modele.estimators_[i].predict_proba(X_test)[:, 1]

df_predictions = df_test[["code_commune"]].copy()
for col, proba in proba_toutes.items():
    df_predictions[col] = proba

# Rattacher le département (depuis df_final_complet)...................................................
df_predictions = df_predictions.merge(
    df_final_complet[["code_commune", "code_departement", "nom_departement"]],
    on="code_commune", how="left"
)

# Agrégation par département (moyenne des probabilités) — table prête pour Looker/BigQuery..............................................
df_predictions_departement = (
    df_predictions.groupby(["code_departement", "nom_departement"])
    [[c for c in df_predictions.columns if c.startswith("proba_")]]
    .mean()
    .round(3)
    .reset_index()
)

print(df_predictions_departement.head(10))
print("\nDimensions :", df_predictions_departement.shape)

In [ ]:
import re

def nettoyer_nom_colonne(nom):
    """Retire accents, espaces et caractères spéciaux pour BigQuery."""
    nom = nom.replace("é", "e").replace("è", "e").replace("ê", "e")
    nom = re.sub(r"[^a-zA-Z0-9_]", "_", nom)
    return nom

# Nettoyer les colonnes des 3 dataframes avant l'export.......................................
df_predictions_departement.columns = [nettoyer_nom_colonne(c) for c in df_predictions_departement.columns]
df_comparaison.columns = [nettoyer_nom_colonne(c) for c in df_comparaison.columns]
df_predictions.columns = [nettoyer_nom_colonne(c) for c in df_predictions.columns]

print("Colonnes department :", df_predictions_departement.columns.tolist())
print("Colonnes comparaison :", df_comparaison.columns.tolist())
print("Colonnes commune :", df_predictions.columns.tolist())

In [ ]:
job_config = bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")

client.load_table_from_dataframe(
    df_predictions_departement,
    "wagon-bootcamp-501621.risques_naturels.predictions_risque_par_departement",
    job_config=job_config
).result()
print("✓ predictions_risque_par_departement")

client.load_table_from_dataframe(
    df_comparaison,
    "wagon-bootcamp-501621.risques_naturels.comparaison_modeles_risque",
    job_config=job_config
).result()
print("✓ comparaison_modeles_risque")

client.load_table_from_dataframe(
    df_predictions,
    "wagon-bootcamp-501621.risques_naturels.predictions_risque_par_commune",
    job_config=job_config
).result()
print("✓ predictions_risque_par_commune")

In [ ]:
query_kpi_cartes = """
SELECT
  type_peril,
  COUNT(DISTINCT code_commune) AS nb_communes_touchees,
  SUM(nb_reconnaissances) AS nb_arretes_total
FROM `sous-nos-pieds.staging.catnat_geo_final_staging`
GROUP BY type_peril
ORDER BY nb_communes_touchees DESC
"""
df_kpi_cartes = client.query(query_kpi_cartes).to_dataframe()
print(df_kpi_cartes)

In [ ]:
job_config = bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")
client.load_table_from_dataframe(
    df_kpi_cartes,
    "wagon-bootcamp-501621.risques_naturels.kpi_communes_touchees_par_alea",
    job_config=job_config
).result()
print("Table kpi_communes_touchees_par_alea exportée")

In [ ]:
query_evolution = """
SELECT
  EXTRACT(YEAR FROM date_debut) AS annee,
  lib_risque_jo,
  COUNT(*) AS nb_arretes
FROM `sous-nos-pieds.staging.catnat_staging`
WHERE date_debut IS NOT NULL
GROUP BY annee, lib_risque_jo
ORDER BY annee
"""

In [ ]:
# Découpage à 3 : train (optimisation du modèle) / val (choix du seuil) / test (évaluation finale, jamais touché avant)
ANNEE_VAL = 2019
ANNEE_TEST = 2022  # ajuste selon ta plage de données

annees_train = range(df_cible_temporelle["annee"].min(), ANNEE_VAL)
annees_val   = range(ANNEE_VAL, ANNEE_TEST)
annees_test  = range(ANNEE_TEST, df_cible_temporelle["annee"].max() + 1)

cible_train = construire_cible(df_cible_temporelle, annees_train)
cible_val   = construire_cible(df_cible_temporelle, annees_val)
cible_test  = construire_cible(df_cible_temporelle, annees_test)

df_train = assembler(df_features, cible_train, colonnes_risque)
df_val   = assembler(df_features, cible_val, colonnes_risque)
df_test  = assembler(df_features, cible_test, colonnes_risque)

print("Train:", df_train.shape, "| Val:", df_val.shape, "| Test:", df_test.shape)

In [ ]:
 echantillon_test = tous_les_codes[:200]
resultats_test = []
with ThreadPoolExecutor(max_workers=3) as executor:
    futures = {executor.submit(get_risques_commune, code): code for code in echantillon_test}
    for i, future in enumerate(as_completed(futures), 1):
        resultats_test.append(future.result())

df_test2 = pd.DataFrame(resultats_test)
print(df_test2["erreur"].value_counts(dropna=False))